# Lesson 2: The Harmonic Oscillator
## From Classical Springs to Quantum Ladders

**Prerequisites:** Lesson 01 (1D Schrödinger solver, eigenvalue problems, finite-difference discretization).

In Lesson 01 we built a general 1D quantum solver and validated it on the particle-in-a-box.
In Lab 01 Exercise 4 we already pointed that solver at a parabolic potential and saw evenly
spaced energy levels appear. Now we understand *why*.

**What you will learn:**

1. **Classical harmonic oscillator** — energy, phase space, classical probability distribution.
2. **Quantum Hamiltonian** — what changes when $x$ and $p$ become operators.
3. **Ladder operators** — an algebraic route to the energy spectrum and eigenstates, without solving any differential equation.
4. **Hermite polynomials** — the differential-equation route, for completeness.
5. **Numerical validation** — reuse our Lesson 01 solver, compare to exact results.
6. **Perturbation theory** — what happens when the potential is *almost* harmonic.

The harmonic oscillator is the single most important model in physics. Every smooth
potential near a minimum looks like a parabola — molecular vibrations, phonons in solids,
electromagnetic field modes, and the starting point for quantum field theory all reduce
to collections of harmonic oscillators.

---
## Part A: The Classical Harmonic Oscillator

### A.1 Hooke's Law and the Potential Parabola

A particle of mass $m$ attached to a spring with force constant $k$ obeys

$$F = -kx \quad \Longrightarrow \quad V(x) = \tfrac{1}{2}kx^2 = \tfrac{1}{2}m\omega^2 x^2,$$

where $\omega = \sqrt{k/m}$ is the angular frequency. The motion is sinusoidal:
$x(t) = A\cos(\omega t + \phi)$.

Why does this simple model appear everywhere? Because **any smooth potential near a
minimum looks like a parabola** to leading order. Taylor-expand $V(x)$ around the
minimum $x_0$:

$$V(x) \approx V(x_0) + \underbrace{V'(x_0)}_{=0}(x-x_0) + \tfrac{1}{2}V''(x_0)(x-x_0)^2 + \cdots$$

The linear term vanishes at a minimum, so the leading behavior is harmonic with
$k = V''(x_0)$.

| System | "Spring constant" $k$ | Typical $\omega$ |
|--------|-----------------------|-----------------|
| Diatomic molecule (e.g. H₂) | Bond curvature $V''(r_e)$ | ~4000 cm⁻¹ |
| Phonon in a crystal | Interatomic force constant | ~10¹³ rad/s |
| LC circuit | $1/LC$ | ~10⁶–10⁹ rad/s |
| Electromagnetic field mode | $\omega^2/c^2$ | any frequency |

**In atomic units** ($\hbar = m_e = e = 1$), with electron mass:

$$\boxed{V(x) = \tfrac{1}{2}\omega^2 x^2}$$

### A.2 Energy Conservation and Phase Space

The total energy of the classical oscillator is

$$E = \frac{p^2}{2m} + \frac{1}{2}m\omega^2 x^2.$$

In atomic units ($m = 1$): $E = p^2/2 + \omega^2 x^2/2$. This is the equation of an
**ellipse** in $(x, p)$ phase space. Each energy $E$ traces out one ellipse; the particle
endlessly orbits it.

Let's plot phase-space ellipses at the energies $E_n = \omega(n + \frac{1}{2})$ that
we'll soon derive quantum mechanically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '.')
from harmonic import HarmonicOscillator, HARTREE_TO_EV, HARTREE_TO_WAVENUMBER

omega = 1.0
ho = HarmonicOscillator(omega)

# --- Phase space plot: each energy E traces an ellipse in (x, p) space ---
fig, ax = plt.subplots(figsize=(6, 6))

# Parametric angle for drawing ellipses (0 to 2π = one full orbit)
theta = np.linspace(0, 2 * np.pi, 300)

colors = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd', '#e377c2']
for n in range(5):
    E = ho.energy(n)

    # From E = p²/2 + ω²x²/2, the phase-space orbit is an ellipse:
    #   x(θ) = x_turn · cos(θ),  where x_turn = √(2E/ω²) (max displacement)
    #   p(θ) = p_max  · sin(θ),  where p_max  = √(2E)     (max momentum)
    x_turn = np.sqrt(2 * E / omega**2)
    p_max = np.sqrt(2 * E)
    x_ellipse = x_turn * np.cos(theta)
    p_ellipse = p_max * np.sin(theta)

    ax.plot(x_ellipse, p_ellipse, color=colors[n], linewidth=1.5,
            label=f'n={n}, E={E:.1f}')

# Axis labels with units (always label units — a course convention)
ax.set_xlabel('x (bohr)', fontsize=12)
ax.set_ylabel('p (ℏ/bohr)', fontsize=12)
ax.set_title('Classical Phase Space — Harmonic Oscillator', fontsize=13)
ax.legend(fontsize=10)
# Equal aspect ratio so circles look circular (the ω=1 case gives circles)
ax.set_aspect('equal')
# Reference axes through the origin
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/phase_space.png', dpi=150, bbox_inches='tight')
plt.show()

Each ellipse is a constant-energy orbit. The particle moves clockwise: rightward
at the top ($p > 0$), leftward at the bottom ($p < 0$). The innermost ellipse
corresponds to $E_0 = \omega/2$ — the quantum ground state energy. Classically,
any energy $E > 0$ is allowed; quantum mechanics picks out only the discrete values
$E_n = \omega(n + \frac{1}{2})$.

### A.3 Classical Probability Distribution

If we repeatedly sample the position of a classical oscillator at random times,
where is it most likely to be found?

The time $dt$ spent in the interval $[x, x+dx]$ is proportional to $1/|v(x)|$,
the reciprocal of the speed. The particle moves **slowest** at the turning points
(where all energy is potential) and **fastest** at the center (where all energy is
kinetic). So the classical probability density **peaks at the edges** and has a
**minimum at the center**.

Quantitatively:

$$P_{\text{cl}}(x) = \frac{1}{\pi \sqrt{x_{\text{turn}}^2 - x^2}}, \qquad |x| < x_{\text{turn}},$$

where $x_{\text{turn}} = \sqrt{2E/\omega^2}$ is the classical turning point.

*Derivation:* From $E = p^2/2 + \omega^2 x^2/2$ we get
$v(x) = p = \sqrt{2E - \omega^2 x^2}$. The fraction of a half-period spent in $dx$ is
$dt/T_{\text{half}} = dx / (v \cdot T_{\text{half}})$, and averaging over both directions
of travel gives the $1/\pi$ normalization.

### A.4 Classical Probability Plots

In [ ]:
# --- Classical probability P(x) for four quantum numbers ---
# Shows how the classical particle spends time: peaks at turning points
# (where it's slowest), minimum at center (where it's fastest).

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
n_values = [0, 2, 5, 10]

for ax, n in zip(axes.flat, n_values):
    # Turning point: where V(x_turn) = E, the particle stops and reverses
    x_turn = ho.turning_point(n)
    # Plot range extends 30% past turning points to show P=0 outside
    x = np.linspace(-x_turn * 1.3, x_turn * 1.3, 1000)
    P_cl = ho.classical_probability(x, n)

    # Probability density curve
    ax.plot(x, P_cl, 'k-', linewidth=1.5)
    # Mark the turning points with vertical dashed lines
    ax.axvline(-x_turn, color='red', linestyle='--', alpha=0.7, label='turning points')
    ax.axvline(x_turn, color='red', linestyle='--', alpha=0.7)
    # Shaded area under the curve to emphasize the distribution shape
    ax.fill_between(x, P_cl, alpha=0.15, color='steelblue')
    ax.set_title(f'n = {n}, E = {ho.energy(n):.1f} ħω', fontsize=11)
    ax.set_xlabel('x (bohr)')
    ax.set_ylabel('P(x)')
    ax.set_ylim(bottom=0)
    if n == 0:
        ax.legend(fontsize=9)

# suptitle sits above the subplots; y=1.02 pushes it above the top edge
fig.suptitle('Classical Probability Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/classical_probability.png', dpi=150, bbox_inches='tight')
plt.show()

The $1/\sqrt{x_{\text{turn}}^2 - x^2}$ singularity at the turning points is the
classical signature: the particle spends the most time where it's momentarily at rest.
Keep this picture in mind — the quantum result will look *qualitatively different* for
low $n$ and only approach this classical limit for large $n$ (the correspondence
principle).

---
## Part B: From Classical to Quantum — What Changes

### B.1 The Quantum Hamiltonian

The classical Hamiltonian $H = p^2/2m + \frac{1}{2}m\omega^2 x^2$ keeps the same
algebraic form in quantum mechanics, but now $x$ and $p$ are **operators**:

$$\hat{H} = \frac{\hat{p}^2}{2m} + \frac{1}{2}m\omega^2 \hat{x}^2.$$

In the position representation, $\hat{x}$ multiplies by $x$ and
$\hat{p} = -i\frac{d}{dx}$ (in atomic units where $\hbar = 1$). The time-independent
Schrödinger equation becomes:

$$-\frac{1}{2}\frac{d^2\phi}{dx^2} + \frac{1}{2}\omega^2 x^2 \phi = E\phi.$$

This is the equation we solved numerically in Lab 01. Now we solve it analytically.

### B.2 The Commutation Relation

The crucial difference between classical and quantum mechanics is that $\hat{x}$ and
$\hat{p}$ **do not commute**:

$$\boxed{[\hat{x}, \hat{p}] \equiv \hat{x}\hat{p} - \hat{p}\hat{x} = i}$$

(In SI units this would be $i\hbar$; in atomic units $\hbar = 1$.)

*Quick derivation:* Apply $[\hat{x}, \hat{p}]$ to a test function $f(x)$:

$$[\hat{x}, \hat{p}]f = x\left(-i\frac{df}{dx}\right) - \left(-i\frac{d}{dx}\right)(xf) = -ix\frac{df}{dx} + i\frac{d(xf)}{dx} = -ix\frac{df}{dx} + if + ix\frac{df}{dx} = if.$$

This single relation is the source of:
- **Heisenberg uncertainty:** $\Delta x \cdot \Delta p \geq \frac{1}{2}$
- **Zero-point energy:** even the ground state has $E_0 = \omega/2 > 0$
- **Discrete energy levels:** the spectrum is quantized

| | Classical | Quantum |
|---|---|---|
| **State** | Point $(x, p)$ in phase space | Wavefunction $\psi(x)$ |
| **$x$ and $p$** | Commuting numbers | Non-commuting operators: $[\hat{x}, \hat{p}] = i$ |
| **Trajectory** | Deterministic orbit | No trajectory — only probability $|\psi|^2$ |
| **Energy** | Continuous, any $E \geq 0$ | Discrete: $E_n = \omega(n + \frac{1}{2})$ |
| **Minimum energy** | $E = 0$ (at rest at $x = 0$) | $E_0 = \omega/2$ (zero-point energy) |
| **Probability at center** | Minimum | Maximum (for $n = 0$) |

### B.3 Natural Units and Dimensional Analysis

The harmonic oscillator has a single parameter $\omega$, which sets both the
**energy scale** and the **length scale** (in atomic units with $m = 1$):

| Quantity | Expression | Physical meaning |
|----------|------------|------------------|
| Energy scale | $E_0 = \omega$ | Spacing between levels |
| Length scale | $x_0 = 1/\sqrt{\omega}$ | Width of ground-state wavefunction |
| Momentum scale | $p_0 = \sqrt{\omega}$ | Ground-state momentum spread |

For a quick estimate: the H₂ molecule has a fundamental vibration at
$\tilde{\nu} \approx 4400$ cm⁻¹, which corresponds to
$\omega \approx 4400/219475 \approx 0.020$ hartree. The ground-state wavefunction
width is $x_0 = 1/\sqrt{0.020} \approx 7$ bohr $\approx 0.37$ Å — comparable to the
bond length. Molecular vibrations are thoroughly quantum mechanical.

> **Habit: Estimate before computing.** Before running any solver, ask: what energy
> scale does $\omega$ set? How wide should the domain be? These estimates catch bugs.

---
## Part C: The Algebraic Approach — Ladder Operators

### C.1 Motivation: Factoring the Hamiltonian

Over real numbers, the sum of squares $a^2 + b^2$ cannot be factored. Over the
**complex numbers**, we can write $a^2 + b^2 = (a + ib)(a - ib)$.

The Hamiltonian $\hat{H} = \hat{p}^2/2 + \omega^2 \hat{x}^2/2$ looks like a sum of
squares. Can we factor it as $(\text{something})(\text{something}^\dagger)$?

Almost — but because $\hat{x}$ and $\hat{p}$ don't commute, the factoring picks up
an extra constant (which turns out to be the zero-point energy). This is the ladder
operator method.

### C.2 Definition of Ladder Operators

Define the **annihilation** (lowering) and **creation** (raising) operators:

$$\boxed{\hat{a} = \frac{1}{\sqrt{2\omega}}\left(\omega\hat{x} + i\hat{p}\right) = \frac{1}{\sqrt{2\omega}}\left(\omega x + \frac{d}{dx}\right)}$$

$$\boxed{\hat{a}^\dagger = \frac{1}{\sqrt{2\omega}}\left(\omega\hat{x} - i\hat{p}\right) = \frac{1}{\sqrt{2\omega}}\left(\omega x - \frac{d}{dx}\right)}$$

where the second form uses $\hat{p} = -i\frac{d}{dx}$.

| Symbol | Name | Action |
|--------|------|--------|
| $\hat{a}$ | Annihilation / lowering | Removes one quantum of energy |
| $\hat{a}^\dagger$ | Creation / raising | Adds one quantum of energy |
| $\hat{N} = \hat{a}^\dagger \hat{a}$ | Number operator | Counts quanta |

### C.3 Hamiltonian in Terms of Ladder Operators

Let's compute $\hat{a}^\dagger \hat{a}$ explicitly:

$$\hat{a}^\dagger \hat{a} = \frac{1}{2\omega}\left(\omega x - \frac{d}{dx}\right)\left(\omega x + \frac{d}{dx}\right).$$

Expanding the product:

$$= \frac{1}{2\omega}\left[\omega^2 x^2 + \omega x \frac{d}{dx} - \frac{d}{dx}(\omega x) - \frac{d^2}{dx^2}\right]$$

The cross terms: $\omega x \frac{d}{dx}$ and $-\frac{d}{dx}(\omega x \cdot) = -\omega - \omega x \frac{d}{dx}$ (product rule). So the cross terms give $-\omega$. Therefore:

$$\hat{a}^\dagger \hat{a} = \frac{1}{2\omega}\left[\omega^2 x^2 - \frac{d^2}{dx^2} - \omega\right] = \frac{1}{\omega}\left[\frac{\omega^2 x^2}{2} + \frac{\hat{p}^2}{2}\right] - \frac{1}{2} = \frac{\hat{H}}{\omega} - \frac{1}{2}.$$

Rearranging:

$$\boxed{\hat{H} = \omega\left(\hat{a}^\dagger \hat{a} + \frac{1}{2}\right) = \omega\left(\hat{N} + \frac{1}{2}\right)}$$

As a bonus, we can also compute $\hat{a}\hat{a}^\dagger$ by repeating the algebra (or noting the sign change in the cross term):

$$\hat{a}\hat{a}^\dagger = \hat{a}^\dagger \hat{a} + 1 = \hat{N} + 1$$

which gives us the fundamental commutator:

$$\boxed{[\hat{a}, \hat{a}^\dagger] = 1}$$

This is the **bosonic commutation relation** — the algebraic backbone of the harmonic oscillator.

### C.4 Ladder Operators as Energy Shifters

What happens when $\hat{a}$ or $\hat{a}^\dagger$ acts on an energy eigenstate?

Using $[\hat{H}, \hat{a}] = -\omega\hat{a}$ (which follows from
$\hat{H}\hat{a} = \omega(\hat{N}+\tfrac{1}{2})\hat{a} = \omega(\hat{a}\hat{N} - \hat{a} + \tfrac{1}{2}\hat{a}) = \hat{a}(\hat{H} - \omega)$):

If $\hat{H}|\phi\rangle = E|\phi\rangle$, then

$$\hat{H}(\hat{a}|\phi\rangle) = (\hat{a}\hat{H} - \omega\hat{a})|\phi\rangle = (E - \omega)(\hat{a}|\phi\rangle).$$

So $\hat{a}|\phi\rangle$ is an eigenstate with energy $E - \omega$. Similarly,
$\hat{a}^\dagger|\phi\rangle$ has energy $E + \omega$.

The operators are a **ladder**: each application of $\hat{a}^\dagger$ steps the energy
up by $\omega$, and each $\hat{a}$ steps it down by $\omega$. This is why the energy
levels are **equally spaced** — a dramatic contrast to the $E_n \propto n^2$ scaling
of the particle-in-a-box (Lesson 01).

### C.5 Ground State from $\hat{a}|0\rangle = 0$

The ladder must have a bottom rung: energies can't be negative (since
$\hat{H} = \hat{p}^2/2 + \omega^2\hat{x}^2/2$ is a sum of squares of Hermitian
operators). There must be a state $|0\rangle$ such that

$$\hat{a}|0\rangle = 0.$$

In position representation, $\hat{a}|0\rangle = 0$ becomes the first-order ODE:

$$\frac{1}{\sqrt{2\omega}}\left(\omega x + \frac{d}{dx}\right)\phi_0(x) = 0 \quad \Longrightarrow \quad \frac{d\phi_0}{dx} = -\omega x \, \phi_0.$$

This is separable: $d\phi_0 / \phi_0 = -\omega x \, dx$, giving
$\phi_0(x) = C \exp(-\omega x^2 / 2)$. Normalizing:

$$\boxed{\phi_0(x) = \left(\frac{\omega}{\pi}\right)^{1/4} \exp\left(-\frac{\omega x^2}{2}\right), \qquad E_0 = \frac{\omega}{2}}$$

The ground state is a **Gaussian** centered at the origin. Its energy $E_0 = \omega/2$
is the **zero-point energy** — a direct consequence of the uncertainty principle.
If the particle were localized at $x = 0$ with $p = 0$, we'd have $\Delta x = \Delta p = 0$,
violating $\Delta x \cdot \Delta p \geq 1/2$. The ground state is the *minimum-uncertainty*
state: $\Delta x = 1/\sqrt{2\omega}$, $\Delta p = \sqrt{\omega/2}$, giving
$\Delta x \cdot \Delta p = 1/2$ exactly.

### C.6 Building Excited States

Starting from $|0\rangle$, we climb the ladder:

$$|n\rangle = \frac{(\hat{a}^\dagger)^n}{\sqrt{n!}} |0\rangle.$$

The normalization factor $1/\sqrt{n!}$ comes from repeatedly applying
$\hat{a}^\dagger |n\rangle = \sqrt{n+1}|n+1\rangle$.

$$\boxed{\begin{aligned}
\hat{a}|n\rangle &= \sqrt{n}\,|n-1\rangle \\
\hat{a}^\dagger|n\rangle &= \sqrt{n+1}\,|n+1\rangle \\
E_n &= \omega\left(n + \tfrac{1}{2}\right), \quad n = 0, 1, 2, \ldots
\end{aligned}}$$

**Key observations:**

1. **Equally spaced levels.** The gap between consecutive levels is always $\omega$,
   independent of $n$. Compare this to the box, where $E_n \propto n^2$ (Lesson 01).

2. **Zero-point energy.** Even $n = 0$ has energy $\omega/2$. You can never remove all
   the energy from a quantum oscillator.

3. **Algebraic structure.** We derived the *entire spectrum* without solving a
   differential equation — just from $[\hat{a}, \hat{a}^\dagger] = 1$ and the
   requirement that energies are bounded below.

### C.7 Forward Reference: Second Quantization

*(This section can be skipped on first reading.)*

The creation/annihilation operators $\hat{a}^\dagger / \hat{a}$ are not just a mathematical
trick — they are the language of **quantum field theory**. The electromagnetic field
decomposes into independent harmonic oscillator modes, each with its own
$\hat{a}^\dagger_k, \hat{a}_k$. A state with $n$ photons in mode $k$ is
$|n_k\rangle = (\hat{a}^\dagger_k)^n/\sqrt{n!}\,|0\rangle$.

In Lessons 6–8, when we reach many-electron quantum chemistry, we'll meet the
**fermionic** creation/annihilation operators $\hat{c}^\dagger_p, \hat{c}_p$, which
satisfy the *anti*commutation relation $\{\hat{c}_p, \hat{c}^\dagger_q\} = \delta_{pq}$.
Same algebraic structure, different statistics.

---
## Part D: The Differential Equation Approach — Hermite Polynomials

### D.1 Hermite Polynomial Solution

The ladder-operator approach gives us the energy spectrum elegantly, but the explicit
spatial form of the wavefunctions comes most naturally from the differential equation route.

Starting from the TISE:

$$-\frac{1}{2}\frac{d^2\phi}{dx^2} + \frac{1}{2}\omega^2 x^2 \phi = E\phi,$$

introduce the dimensionless coordinate $\xi = \sqrt{\omega}\, x$ (equivalently, $\alpha = \sqrt{\omega}$). The equation becomes:

$$\frac{d^2\phi}{d\xi^2} + \left(\frac{2E}{\omega} - \xi^2\right)\phi = 0.$$

**Asymptotic analysis:** For $|\xi| \to \infty$, the $\xi^2$ term dominates, and
the solutions behave as $e^{\pm\xi^2/2}$. Only $e^{-\xi^2/2}$ is normalizable.

**Factoring out the Gaussian:** Write $\phi(\xi) = H(\xi)\,e^{-\xi^2/2}$. Substituting
gives **Hermite's equation:**

$$H'' - 2\xi H' + (2E/\omega - 1)H = 0.$$

A power-series solution terminates (giving a normalizable wavefunction) only when
$2E/\omega - 1 = 2n$ for non-negative integer $n$, i.e., $E_n = \omega(n + 1/2)$ — the
same result as the algebraic approach. The terminating polynomials are the **Hermite
polynomials** $H_n(\xi)$.

The normalized wavefunctions are:

$$\boxed{\phi_n(x) = \left(\frac{\alpha}{\sqrt{\pi}\, 2^n\, n!}\right)^{1/2} H_n(\alpha x)\, e^{-\alpha^2 x^2/2}, \qquad \alpha = \sqrt{\omega}}$$

The first few Hermite polynomials:

| $n$ | $H_n(\xi)$ | Nodes |
|-----|-----------|-------|
| 0 | $1$ | 0 |
| 1 | $2\xi$ | 1 |
| 2 | $4\xi^2 - 2$ | 2 |
| 3 | $8\xi^3 - 12\xi$ | 3 |
| 4 | $16\xi^4 - 48\xi^2 + 12$ | 4 |

In our code, we use `scipy.special.eval_hermite(n, xi)` for stable evaluation up to
$n \sim 50$.

### D.2 Connection Between Approaches

The algebraic (ladder operator) approach and the differential equation (Hermite polynomial)
approach arrive at the same destination from different directions. The algebra gives the
**energy structure** — equally spaced levels, zero-point energy — with minimal computation.
The differential equation gives the **spatial structure** — explicit wavefunctions as
Gaussian-weighted polynomials. Both perspectives are essential.

---
## Part E: From Theory to Simulation

### E.1 Domain Choice and Setup

Unlike the infinite square well, the harmonic oscillator has **soft walls** — the
wavefunction decays as a Gaussian into the classically forbidden region but never
exactly reaches zero. Our finite-difference solver (from Lesson 01) imposes hard-wall
boundary conditions $\phi(x_{\min}) = \phi(x_{\max}) = 0$, so we must choose a domain
large enough that the wavefunctions are negligible at the boundaries.

The `suggest_domain(n_max)` method extends several characteristic lengths $1/\sqrt{\omega}$
beyond the classical turning point of the highest state. Too small a domain truncates
the wavefunction tails; too large wastes grid points on zeros.

In [ ]:
# --- Set up the harmonic oscillator for numerical solution ---
# We reuse the Lesson 01 solver, which knows nothing about the HO specifically.
# It just discretizes -½ d²φ/dx² + V(x)φ = Eφ on a grid.

sys.path.insert(0, '../lesson-01-particle-in-a-box')
from quantum1d import QuantumSystem1D

omega = 1.0
ho = HarmonicOscillator(omega)
n_states = 8  # how many energy levels we want

# Domain choice: extend past the classical turning points of the highest state.
# suggest_domain returns (x_min, x_max) with padding for the Gaussian tails.
# n_sigma=4.0 means we go 4 characteristic lengths (1/√ω) beyond the turning point.
x_min, x_max = ho.suggest_domain(n_states - 1, n_sigma=4.0)
N = 400  # number of interior grid points (boundary points are fixed at φ=0)

print(f'ω = {omega} hartree')
print(f'Domain: [{x_min:.2f}, {x_max:.2f}] bohr')
print(f'Classical turning point for n={n_states-1}: ±{ho.turning_point(n_states-1):.2f} bohr')
print(f'Grid: {N} interior points, dx = {(x_max - x_min)/(N+1):.4f} bohr')

# Create the system: V(x) = ½ω²x² is the only physics input.
# QuantumSystem1D builds the Hamiltonian matrix H = T + V internally.
system = QuantumSystem1D(x_min, x_max, N,
                         V_func=lambda x: 0.5 * omega**2 * x**2)

# solve() calls scipy.linalg.eigh to diagonalize H, then normalizes
# the eigenvectors on the grid (so that Σ|φ|²·dx = 1).
energies, states = system.solve(n_states)

### E.2 Energy Comparison Table

How accurate are our numerical energies? More importantly — **how accurate do they need
to be?** The answer depends on the physical question:

- **Spectroscopy** measures energy *differences* (e.g., the $n=0 \to 1$ transition
  frequency). For molecular vibrations, experimental precision is ~0.1 cm⁻¹ out of
  ~4000 cm⁻¹, or about 1 part in 10⁴. Our solver needs to match this.

- **Thermodynamics** requires energies accurate to $\sim k_B T$. At room temperature,
  $k_B T \approx 0.001$ hartree, so errors of $10^{-4}$ hartree matter; errors of
  $10^{-6}$ hartree don't.

The table below shows relative errors — the ratio $|E_{\text{num}} - E_{\text{exact}}|/E_{\text{exact}}$.
A relative error of $10^{-4}$ means the energy is correct to 4 significant figures.

In [ ]:
# --- Energy comparison: numerical vs. exact ---
# Print a formatted table showing how well our finite-difference solver
# reproduces the analytical result E_n = ω(n+½).

print(f'{"n":>3s}  {"E_numerical":>14s}  {"E_exact":>14s}  {"Rel. error":>12s}')
print('-' * 50)
for n in range(n_states):
    E_num = energies[n]
    E_exact = ho.energy(n)
    # Relative error: |E_num - E_exact| / E_exact
    # This tells us how many significant figures are correct.
    # 1e-4 → 4 correct digits; 1e-6 → 6 correct digits.
    rel_err = abs(E_num - E_exact) / E_exact
    print(f'{n:3d}  {E_num:14.8f}  {E_exact:14.8f}  {rel_err:12.2e}')

# --- Translate the error into physically measurable units ---
# A spectroscopist measures transition frequencies, not absolute energies.
# Convert the error in the 0→1 transition to wavenumbers (cm⁻¹).
dE_01_num = energies[1] - energies[0]
dE_01_exact = ho.energy(1) - ho.energy(0)
transition_error_cm = abs(dE_01_num - dE_01_exact) * HARTREE_TO_WAVENUMBER
print(f'\n--- Physical context ---')
print(f'Transition 0→1: {dE_01_num * HARTREE_TO_WAVENUMBER:.1f} cm⁻¹'
      f' (exact: {dE_01_exact * HARTREE_TO_WAVENUMBER:.1f} cm⁻¹)')
print(f'Error in transition frequency: {transition_error_cm:.1f} cm⁻¹')
print(f'For comparison: typical experimental precision is ~0.1 cm⁻¹')

### E.3 Wavefunction Comparison

In [ ]:
# --- Wavefunction comparison: numerical solver vs. exact Hermite-Gaussian ---
# Two-panel figure: left = wavefunctions, right = probability densities.
# Each state is offset vertically by its energy, so the plot also shows
# the energy level diagram.

n_show = 5
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

# x_full: grid including boundary points (where φ=0 by our boundary condition)
x_full = system.x_full()
# x_fine: denser grid for plotting smooth exact wavefunctions
x_fine = np.linspace(x_min, x_max, 1000)

# Draw the parabolic potential V(x) = ½ω²x² as a light gray background
# to show where the "walls" of the well are
V_bg = 0.5 * omega**2 * x_fine**2
for ax in (ax1, ax2):
    ax.fill_between(x_fine, 0, V_bg, alpha=0.08, color='gray')
    ax.plot(x_fine, V_bg, 'k-', linewidth=0.5, alpha=0.3)

colors = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd', '#e377c2']

for n in range(n_show):
    E = ho.energy(n)
    color = colors[n % len(colors)]

    # --- Sign fix for numerical eigenvectors ---
    # The eigensolver returns eigenvectors with arbitrary overall sign (±φ are
    # both valid eigenstates). We fix the sign so the numerical curve overlaps
    # the exact one: if the dot product is negative, the sign is flipped.
    phi_num = system.state_full(n)
    phi_exact_at_grid = ho.wavefunction(x_full, n)
    if np.dot(phi_num, phi_exact_at_grid) < 0:
        phi_num = -phi_num

    # Exact wavefunction evaluated on the fine grid (smooth curve)
    phi_exact = ho.wavefunction(x_fine, n)

    # Left panel: wavefunctions φ(x), each offset by energy E_n
    # Solid = numerical (from finite-difference solver)
    # Dashed = exact (from Hermite-Gaussian formula)
    ax1.plot(x_full, phi_num + E, color=color, linewidth=1.5,
             label=f'n={n}, E={E:.2f}' if n < 5 else None)
    ax1.plot(x_fine, phi_exact + E, color=color, linewidth=1.0,
             linestyle='--', alpha=0.7)
    ax1.axhline(E, color='gray', linewidth=0.3)  # energy level line

    # Right panel: probability densities |φ(x)|²
    # fill_between creates shaded area showing where the particle is likely found
    ax2.fill_between(x_full, E, phi_num**2 + E, alpha=0.15, color=color)
    ax2.plot(x_full, phi_num**2 + E, color=color, linewidth=1.5)
    ax2.plot(x_fine, phi_exact**2 + E, color=color, linewidth=1.0,
             linestyle='--', alpha=0.7)
    ax2.axhline(E, color='gray', linewidth=0.3)

# Formatting
ax1.set_xlabel('x (bohr)', fontsize=12)
ax1.set_ylabel('φ(x) + E (hartree)', fontsize=12)
ax1.set_title('Wavefunctions (solid: numerical, dashed: exact)', fontsize=11)
ax1.legend(fontsize=9)
ax1.set_ylim(-0.5, ho.energy(n_show - 1) + 2.0)

ax2.set_xlabel('x (bohr)', fontsize=12)
ax2.set_ylabel('|φ(x)|² + E', fontsize=12)
ax2.set_title('Probability Densities', fontsize=11)
ax2.set_ylim(-0.5, ho.energy(n_show - 1) + 2.0)

plt.tight_layout()
plt.savefig('figures/ho_eigenstates.png', dpi=150, bbox_inches='tight')
plt.show()

The numerical (solid) and exact (dashed) curves are indistinguishable at this grid
resolution. State $n$ has exactly $n$ nodes, and each energy level is separated by
$\omega = 1$ hartree — the equally-spaced ladder predicted by our algebraic derivation.

### E.4 Classical vs. Quantum Probability — The Key Figure

Now for the money plot: how does the quantum probability density $|\phi_n(x)|^2$
compare to the classical prediction $P_{\text{cl}}(x)$?

In [ ]:
# --- The key figure: quantum |φ_n|² vs classical P_cl(x) ---
# This is the correspondence principle in action: quantum → classical for large n.

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
n_values = [0, 2, 5, 10]

# For n=10 we need a larger domain and more grid points.
# Recompute with enough states to include n=10.
x_min_big, x_max_big = ho.suggest_domain(12, n_sigma=4.0)
system_big = QuantumSystem1D(x_min_big, x_max_big, 600,
                              V_func=lambda x: 0.5 * omega**2 * x**2)
system_big.solve(n_states=12)

for ax, n in zip(axes.flat, n_values):
    x_turn = ho.turning_point(n)
    # Plot range: 1.5× the turning point to show tunneling tails
    x_plot = np.linspace(-x_turn * 1.5, x_turn * 1.5, 1000)

    # Classical: P(x) = 1/(π√(x_turn²-x²)), peaks at turning points
    P_cl = ho.classical_probability(x_plot, n)

    # Quantum: |φ_n(x)|² from exact Hermite-Gaussian formula
    phi = ho.wavefunction(x_plot, n)
    P_qm = phi**2

    # Plot both on the same axes for direct comparison
    ax.plot(x_plot, P_qm, '#1f77b4', linewidth=1.5, label='Quantum $|\\phi_n|^2$')
    ax.plot(x_plot, P_cl, 'r--', linewidth=1.5, label='Classical $P_{cl}$')
    # Dotted lines mark the classical turning points
    ax.axvline(-x_turn, color='gray', linestyle=':', alpha=0.5)
    ax.axvline(x_turn, color='gray', linestyle=':', alpha=0.5)
    # Light shading under the quantum curve
    ax.fill_between(x_plot, P_qm, alpha=0.1, color='#1f77b4')

    ax.set_title(f'n = {n}', fontsize=12)
    ax.set_xlabel('x (bohr)')
    ax.set_ylabel('Probability density')
    ax.set_ylim(bottom=0)
    if n == 0:
        ax.legend(fontsize=9)

fig.suptitle('Quantum vs. Classical Probability — The Correspondence Principle',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/classical_vs_quantum.png', dpi=150, bbox_inches='tight')
plt.show()

**What to notice:**

- **$n = 0$:** The quantum and classical distributions are *qualitatively opposite*.
  The quantum ground state peaks at the center (Gaussian); the classical distribution
  peaks at the turning points.

- **$n = 2, 5$:** The quantum oscillations begin to cluster toward the turning points,
  but the individual peaks and nodes are still distinctly quantum.

- **$n = 10$:** The envelope of $|\phi_n|^2$ closely follows the classical $P_{\text{cl}}(x)$.
  The quantum oscillations are rapid and, if averaged over, reproduce the classical
  distribution. This is the **correspondence principle** in action: large quantum numbers
  approach classical behavior.

- **Beyond the turning points:** Quantum mechanics always predicts a nonzero probability
  in the classically forbidden region — **tunneling into the classically forbidden region**.
  This probability decays exponentially.

### E.5 Validation: Orthonormality and Node Counting

Beyond matching energies, we need to verify the **wavefunctions** themselves are
physically correct. Two checks:

**Orthonormality** — $\langle \phi_m | \phi_n \rangle = \delta_{mn}$. This is not just a
mathematical nicety: it's the statement that **measuring the system in state $n$ will never
find it in state $m$** (for $m \neq n$). If the off-diagonal overlaps were, say, $10^{-2}$,
our "eigenstates" would be contaminated — a 1% probability of misidentifying one state as
another. At $10^{-15}$, the contamination is at machine-precision level, meaning our
discretization preserves the orthogonality of the underlying continuous eigenstates.

**Node counting** — State $n$ must have exactly $n$ nodes (zero crossings). This is a
topological property guaranteed by Sturm-Liouville theory. If the node count is wrong,
we've either missed a state or the grid is too coarse to resolve the oscillations.

In [ ]:
# --- Orthonormality check ---
# ⟨φ_m|φ_n⟩ should be the identity matrix (δ_mn).
# Physically: states are perfectly distinguishable. Off-diagonal ≠ 0 would mean
# our "state n" is contaminated with other states.
n_check = 6
overlap = system.check_orthonormality(n_check)

print('Overlap matrix ⟨φ_m|φ_n⟩ (should be δ_mn):')
print()
# Print as a formatted table with row/column indices
header = '     ' + ''.join(f'{n:>10d}' for n in range(n_check))
print(header)
for m in range(n_check):
    row = f'{m:>3d}  ' + ''.join(f'{overlap[m, n]:10.2e}' for n in range(n_check))
    print(row)

# Report the worst off-diagonal element as a single quality metric
max_off_diag = max(abs(overlap[m, n])
                   for m in range(n_check)
                   for n in range(n_check) if m != n)
print(f'\nMax off-diagonal element: {max_off_diag:.2e}')
print(f'  (Machine epsilon ≈ 2e-16 — our orthogonality is at machine precision.)')
print(f'  (If this were ~1e-2, we would have ~1% state contamination — unacceptable.)')

# --- Node counting ---
# State n must have exactly n nodes (zero crossings).
# This is a topological invariant — if the count is wrong, something is broken.
print(f'\n{"State n":>8s}  {"Expected nodes":>15s}  {"Counted nodes":>14s}  {"Match":>6s}')
print('-' * 50)
for n in range(n_check):
    # count_nodes counts sign changes in the wavefunction array
    nodes = system.count_nodes(n)
    match = '✓' if nodes == n else '✗'
    print(f'{n:8d}  {n:15d}  {nodes:14d}  {match:>6s}')

### E.6 Convergence Study

For the harmonic oscillator, our numerical solution has **two independent error sources:**

1. **Discretization error** — finite grid spacing $dx$. Converges as $\mathcal{O}(dx^2)$
   for our second-order finite differences.
2. **Domain truncation error** — imposing $\phi = 0$ at finite boundaries. Converges
   exponentially as we extend the domain past the turning points.

We study each independently by fixing the other.

> **Translation-layer lesson:** When your solver has multiple knobs ($N$ and domain size),
> you need to understand which error source dominates. Increasing $N$ doesn't help if the
> domain is too small (you're just resolving the *wrong* boundary condition more finely).
> Expanding the domain doesn't help if $N$ is too small (you're spreading grid points over
> zeros instead of resolving the wavefunction). Always identify and control *each* error
> source independently.

In [ ]:
# --- Convergence study: two independent error sources ---
# Left panel: how error shrinks as we add more grid points (fixed domain).
# Right panel: how error shrinks as we widen the domain (fixed grid density).

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ====== Study 1: Grid resolution (fixed domain) ======
# Use a generous domain so boundary truncation is negligible.
# Then the only error source is the finite-difference approximation d²φ/dx².
x_min_conv, x_max_conv = ho.suggest_domain(5, n_sigma=5.0)
N_values = [50, 100, 200, 400, 800, 1600]
errors_N = []

for N in N_values:
    sys_conv = QuantumSystem1D(x_min_conv, x_max_conv, N,
                               V_func=lambda x: 0.5 * omega**2 * x**2)
    E_conv, _ = sys_conv.solve(6)
    # Track the ground-state error (most sensitive to grid quality)
    errors_N.append(abs(E_conv[0] - ho.energy(0)) / ho.energy(0))

# Log-log plot: if error ~ 1/N², the points fall on a line with slope -2
ax1.loglog(N_values, errors_N, 'bo-', linewidth=1.5, markersize=6)
# Reference slope line: O(N⁻²) for second-order finite differences
N_ref = np.array([50, 1600])
ax1.loglog(N_ref, 0.5 * (N_ref[0] / N_ref)**2 * errors_N[0],
           'k--', alpha=0.5, label='$\\mathcal{O}(N^{-2})$')
ax1.set_xlabel('Number of grid points N', fontsize=12)
ax1.set_ylabel('Relative error in $E_0$', fontsize=12)
ax1.set_title('Convergence with grid resolution\n(fixed domain)', fontsize=11)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# ====== Study 2: Domain size (fixed N) ======
# Use a fixed grid density, but vary how far the domain extends past
# the turning points. The Gaussian tail φ ~ exp(-ωx²/2) means the
# truncation error drops exponentially with domain size.
N_fixed = 400
sigma_values = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0]
errors_domain = []

for ns in sigma_values:
    x_min_d, x_max_d = ho.suggest_domain(5, n_sigma=ns)
    sys_d = QuantumSystem1D(x_min_d, x_max_d, N_fixed,
                            V_func=lambda x: 0.5 * omega**2 * x**2)
    E_d, _ = sys_d.solve(6)
    err = abs(E_d[0] - ho.energy(0)) / ho.energy(0)
    # Floor at 1e-16 to avoid log(0) on the semilogy plot
    errors_domain.append(max(err, 1e-16))

# Semilogy: exponential convergence appears as a straight line
ax2.semilogy(sigma_values, errors_domain, 'rs-', linewidth=1.5, markersize=6)
ax2.set_xlabel('Domain extension (n_sigma)', fontsize=12)
ax2.set_ylabel('Relative error in $E_0$', fontsize=12)
ax2.set_title('Convergence with domain size\n(fixed N=400)', fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/ho_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print('Domain truncation error drops exponentially (Gaussian tail).')
print('Discretization error drops algebraically as O(1/N²).')
print('Both must be controlled for accurate results.')

---
## Part F: Perturbation Theory — When Exact Solutions Run Out

### F.1 The Anharmonic Potential

Real molecular potentials are not perfectly parabolic. The simplest correction is a
quartic term:

$$V(x) = \frac{1}{2}\omega^2 x^2 + \lambda x^4.$$

When $\lambda$ is small, we can treat $\hat{H}' = \lambda x^4$ as a **perturbation**
to the harmonic oscillator and compute corrections analytically.

### F.2 First-Order Perturbation Theory

Write the Hamiltonian as $\hat{H} = \hat{H}_0 + \hat{H}'$, where $\hat{H}_0$ is
the harmonic oscillator with known eigenstates $|n\rangle$ and energies $E_n^{(0)} = \omega(n + \frac{1}{2})$.

First-order perturbation theory gives the energy correction:

$$E_n^{(1)} = \langle n | \hat{H}' | n \rangle = \lambda \langle n | x^4 | n \rangle.$$

To compute $\langle n | x^4 | n \rangle$, we express $\hat{x}$ in terms of ladder operators.

### F.3 Computing $\langle n | x^4 | n \rangle$ via Ladder Operators

From the definitions of $\hat{a}$ and $\hat{a}^\dagger$, we can invert to get:

$$\hat{x} = \frac{1}{\sqrt{2\omega}}(\hat{a}^\dagger + \hat{a}).$$

Therefore:

$$\hat{x}^4 = \frac{1}{(2\omega)^2}(\hat{a}^\dagger + \hat{a})^4.$$

We need the diagonal matrix element $\langle n | (\hat{a}^\dagger + \hat{a})^4 | n \rangle$.
When we expand $(\hat{a}^\dagger + \hat{a})^4$, only terms with **equal numbers of
$\hat{a}^\dagger$ and $\hat{a}$** contribute to the diagonal (others change the state
and give zero by orthogonality). The contributing terms are those with two raising and
two lowering operators in various orderings.

After careful normal-ordering (moving all $\hat{a}^\dagger$ to the left using
$[\hat{a}, \hat{a}^\dagger] = 1$), the result is:

$$\langle n | (\hat{a}^\dagger + \hat{a})^4 | n \rangle = 6n^2 + 6n + 3.$$

Therefore:

$$\boxed{\langle n | x^4 | n \rangle = \frac{3}{4\omega^2}(2n^2 + 2n + 1)}$$

and the first-order energy correction is:

$$\boxed{E_n^{(1)} = \frac{3\lambda}{4\omega^2}(2n^2 + 2n + 1)}$$

Note that the correction grows as $n^2$ — higher states are more affected because
$\langle x^4 \rangle$ grows with the amplitude of oscillation.

### F.4 Perturbation Theory vs. Numerical Solver

Let's compare the first-order PT prediction to a full numerical diagonalization
for a range of $\lambda$ values.

In [ ]:
# --- Perturbation theory vs. full numerical diagonalization ---
# For V(x) = ½ω²x² + λx⁴, compare the 1st-order PT prediction
# E_n ≈ E_n^(0) + λ⟨n|x⁴|n⟩ against the exact numerical answer.

lambda_values = [0, 0.01, 0.02, 0.05, 0.1, 0.2]
n_show_pt = 4  # compare states n=0,1,2,3

# Storage arrays: rows = λ values, columns = quantum numbers
E_numerical = np.zeros((len(lambda_values), n_show_pt))
E_pt = np.zeros((len(lambda_values), n_show_pt))

# Use a generous domain (the quartic term confines more, so this is safe)
x_min_pt, x_max_pt = ho.suggest_domain(5, n_sigma=5.0)

for i, lam in enumerate(lambda_values):
    # --- Numerical: solve the full anharmonic problem from scratch ---
    # Note the lam=lam default argument: this avoids a Python closure bug
    # where all lambdas would capture the same loop variable.
    sys_pt = QuantumSystem1D(
        x_min_pt, x_max_pt, 500,
        V_func=lambda x, lam=lam: 0.5 * omega**2 * x**2 + lam * x**4
    )
    E_num, _ = sys_pt.solve(n_show_pt + 2)
    E_numerical[i, :] = E_num[:n_show_pt]

    # --- Perturbation theory: E_n ≈ ω(n+½) + λ·(3/4ω²)(2n²+2n+1) ---
    for n in range(n_show_pt):
        E_pt[i, n] = ho.energy(n) + ho.perturbation_energy_first_order(n, lam)

# --- Plot: E_n(λ) for each state, numerical vs. PT ---
fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd']

for n in range(n_show_pt):
    # Circles + solid line: numerical (exact for this grid)
    ax.plot(lambda_values, E_numerical[:, n], 'o-', color=colors[n],
            linewidth=1.5, markersize=5, label=f'n={n} (numerical)')
    # Squares + dashed line: perturbation theory (approximate)
    ax.plot(lambda_values, E_pt[:, n], 's--', color=colors[n],
            linewidth=1.0, markersize=4, alpha=0.7,
            label=f'n={n} (1st-order PT)')

ax.set_xlabel('λ (perturbation strength)', fontsize=12)
ax.set_ylabel('Energy (hartree)', fontsize=12)
ax.set_title('Anharmonic Oscillator: Perturbation Theory vs. Exact', fontsize=13)
# Two-column legend: numerical and PT labels side by side
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/anharmonic_perturbation.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Print comparison tables at selected λ values ---
for lam_idx in [1, 3, 5]:  # λ = 0.01, 0.05, 0.2
    lam = lambda_values[lam_idx]
    print(f'\nλ = {lam}:')
    print(f'{"n":>3s}  {"E_numerical":>12s}  {"E_PT":>12s}  {"Difference":>12s}')
    print('-' * 45)
    for n in range(n_show_pt):
        diff = E_numerical[lam_idx, n] - E_pt[lam_idx, n]
        print(f'{n:3d}  {E_numerical[lam_idx, n]:12.6f}  {E_pt[lam_idx, n]:12.6f}  {diff:12.6f}')

### F.5 When Perturbation Theory Breaks Down

First-order perturbation theory is valid when the correction is small compared to
the unperturbed energy spacing:

$$|E_n^{(1)}| \ll \omega.$$

Since $E_n^{(1)} \propto \lambda n^2 / \omega^2$, perturbation theory breaks down for:
- **Large $\lambda$** — the perturbation is too strong.
- **Large $n$** — higher states are more sensitive because the particle explores
  larger $|x|$ where $x^4$ dominates.

In the plot above, PT works well for $\lambda \leq 0.02$ (especially for low $n$),
but diverges significantly from the exact result by $\lambda = 0.2$.

The numerical solver, on the other hand, handles *any* $\lambda$ — this is the payoff
of having a validated computational tool. When analytical methods fail, the solver
keeps working.

---
## Summary

### What We Built

| Component | What it does |
|-----------|-------------|
| Exact energies | $E_n = \omega(n + \frac{1}{2})$ — derived algebraically |
| Exact wavefunctions | Hermite-Gaussian $\phi_n(x)$ — derived from ODE |
| Numerical solver | Reused from Lesson 01, validated against exact results |
| Classical probability | $P_{\text{cl}}(x) = 1/(\pi\sqrt{x_t^2 - x^2})$ — for comparison |
| Perturbation theory | First-order correction for $V = \frac{1}{2}\omega^2 x^2 + \lambda x^4$ |

### Key Takeaways

1. **The harmonic oscillator is universal.** Any potential near a minimum is approximately
   harmonic. Understanding this one system unlocks molecular vibrations, phonons,
   photon modes, and much more.

2. **Ladder operators reveal algebraic structure.** The entire energy spectrum follows
   from $[\hat{a}, \hat{a}^\dagger] = 1$ — no differential equations needed.

3. **Two error sources for soft-wall problems.** Discretization error (controlled by $N$)
   and domain truncation error (controlled by domain size) are independent and both
   must be managed. Increasing one knob doesn't fix the other.

4. **The correspondence principle works.** For large $n$, quantum probability
   distributions approach the classical prediction — but for small $n$, quantum and
   classical behavior are qualitatively different.

5. **Perturbation theory extends analytical tools.** When the exact solution is known
   for a nearby problem, PT gives corrections analytically — but fails when the
   perturbation is too strong. The numerical solver has no such limitation.

6. **Know your error budget.** A relative error of $10^{-4}$ in energy sounds small,
   but for spectroscopy it translates to ~10 cm⁻¹ uncertainty — larger than typical
   experimental precision. Always convert numerical errors to physical units relevant
   to the measurement you're modeling.

### What Comes Next

In **Lesson 03**, we move to higher dimensions: the 2D/3D harmonic oscillator, separation
of variables, angular momentum (where $\hat{L}^+$ and $\hat{L}^-$ are ladder operators
for angular momentum, just as $\hat{a}^\dagger$ and $\hat{a}$ are for energy), and
ultimately the hydrogen atom.

---
## Exercises

Reference solutions are available in [lab-02-exercises.ipynb](lab-02-exercises.ipynb).

---

**Exercise 1: Zero-Point Energy from the Uncertainty Principle**

Compute $\Delta x$ and $\Delta p$ for the ground state $\phi_0(x)$ of the harmonic
oscillator (analytically using the Gaussian form, and numerically from the grid).
Verify that $\Delta x \cdot \Delta p = \frac{1}{2}$ — the minimum allowed by the
uncertainty principle. Use this to re-derive $E_0 = \omega/2$.

---

**Exercise 2: Mass Dependence — Diatomic Vibration**

For a diatomic molecule, the oscillating "particle" has the reduced mass
$\mu = m_1 m_2 / (m_1 + m_2)$. Using $\omega = \sqrt{k/\mu}$:

(a) Show that the vibrational frequency scales as $\omega \propto 1/\sqrt{\mu}$.

(b) For HCl ($k \approx 480$ N/m), compute the fundamental frequency in cm⁻¹ and
compare to the experimental value of ~2886 cm⁻¹. Use `HARTREE_TO_WAVENUMBER`.

(c) What happens if you replace H with D (deuterium)? Compute the isotope shift.

---

**Exercise 3: Coherent States — The "Most Classical" Quantum State**

A coherent state $|\alpha\rangle$ is an eigenstate of the annihilation operator:
$\hat{a}|\alpha\rangle = \alpha|\alpha\rangle$, where $\alpha \in \mathbb{C}$.
Its expansion in energy eigenstates is:

$$|\alpha\rangle = e^{-|\alpha|^2/2} \sum_{n=0}^{\infty} \frac{\alpha^n}{\sqrt{n!}} |n\rangle.$$

(a) Build the coherent state wavefunction $\psi(x, 0) = \langle x | \alpha \rangle$
by summing enough terms (truncate when contributions are negligible). Use $\alpha = 2$.

(b) Time-evolve: $\psi(x, t) = e^{-|\alpha|^2/2} \sum_n \frac{\alpha^n}{\sqrt{n!}} \phi_n(x) e^{-iE_n t}$.
Plot $|\psi(x,t)|^2$ at several times in one period $T = 2\pi/\omega$.

(c) Does the probability density oscillate like a classical particle?

---

**Exercise 4: Ladder Operator Matrix Elements**

Compute the following matrix elements using ladder operators:

(a) $\langle m | \hat{x} | n \rangle$   (selection rule: $\Delta n = \pm 1$)

(b) $\langle m | \hat{p} | n \rangle$

(c) $\langle m | \hat{x}^2 | n \rangle$   (selection rule: $\Delta n = 0, \pm 2$)

Verify your results numerically by constructing $x_{mn} = \int \phi_m(x) \, x \, \phi_n(x) \, dx$
on the grid.

---

**Exercise 5: Second-Order Perturbation Theory**

The second-order energy correction is:

$$E_n^{(2)} = \sum_{m \neq n} \frac{|\langle m | \hat{H}' | n \rangle|^2}{E_n^{(0)} - E_m^{(0)}}.$$

For $\hat{H}' = \lambda x^4$ and $n = 0$:

(a) Which states $|m\rangle$ contribute? (Hint: $x^4$ connects $n$ to $n \pm 2$ and $n \pm 4$.)

(b) Compute $E_0^{(2)}$ numerically by evaluating the sum for $\lambda = 0.05$.

(c) Compare $E_0^{(0)} + E_0^{(1)} + E_0^{(2)}$ to the exact numerical value.
How much does the second-order correction improve the result?

---

**Exercise 6: The Morse Potential — Real Molecular Vibrations**

The Morse potential models a real diatomic bond:

$$V(r) = D_e \left(1 - e^{-a(r - r_e)}\right)^2,$$

where $D_e$ is the dissociation energy, $r_e$ is the equilibrium bond length, and
$a = \omega\sqrt{\mu/(2D_e)}$.

For H₂: $D_e = 0.1745$ hartree, $r_e = 1.401$ bohr, $\mu = 918.1$ (half proton mass
in electron masses).

(a) Set up the Morse potential and find the bound states numerically.

(b) Compare the lowest few energy spacings to the harmonic approximation
$\Delta E = \omega$ (where $\omega = a\sqrt{2D_e/\mu}$).

(c) Convert the fundamental frequency to cm⁻¹ and compare to the experimental
value of 4161 cm⁻¹.